In [2]:

import json, os, shutil, yaml
from pathlib import Path
from PIL import Image
import numpy as np
from tqdm import tqdm


root = Path(".").resolve()

ann_dir = root / "annotations"
img_dir = root / "images"

splits = {"train": "train.json",
          "val":   "val.json",
          "test":  "test.json"}

labels_root = root / "labels"
labels_root.mkdir(exist_ok=True)


#discover classes
class_names = []
for ann_file in ann_dir.glob("*.json"):
    data = json.load(open(ann_file))
    for s in data["shapes"]:
        lab = s["label"]
        if lab not in class_names:
            class_names.append(lab)
name2id = {n: i for i, n in enumerate(class_names)}
print("Detected classes:", class_names)


def convert_one(labelme_json, w, h):
    """
    Convert one LabelMe JSON dict to a list of YOLO‑v8 seg lines.
    Each line:  cls  xc  yc  bw  bh  x1 y1 x2 y2 … xn yn   (all 0‑1)
    """
    yolo_lines = []
    for s in labelme_json["shapes"]:
        pts = np.array(s["points"], dtype=float)
        xmin, ymin = pts.min(axis=0)
        xmax, ymax = pts.max(axis=0)
        xc = ((xmin + xmax) / 2) / w
        yc = ((ymin + ymax) / 2) / h
        bw = (xmax - xmin) / w
        bh = (ymax - ymin) / h
        # polygon points normalised
        poly = " ".join(f"{p:.6f}" for p in (pts / [w, h]).flatten())
        cls_id = name2id[s["label"]]
        yolo_lines.append(f"{cls_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f} {poly}")
    return "\n".join(yolo_lines)

for split, list_file in splits.items():
    split_path = root / list_file
    if not split_path.exists():
        print(f"[WARN] {list_file} not found – skipping {split} split.")
        continue

    print(f"\n▶ Preparing {split} split from {list_file}")
    with open(split_path) as f:
        content = json.load(f)

    if isinstance(content, list):
        img_names = content
    elif isinstance(content, dict):
        if "images" in content:
            img_names = [img["file_name"] for img in content["images"]]
        else:
            raise ValueError(
                f"{list_file} is a dict but does not contain an 'images' key; "
                f"please adapt the loader."
            )
    else:
        raise ValueError(f"{list_file} must be JSON list or dict, not {type(content)}")

    print(f"{split}: {len(img_names)} images – first 5:", img_names[:5])

    # Ensure destination folders exist
    (root / "images" / split).mkdir(parents=True, exist_ok=True)
    (labels_root / split).mkdir(parents=True, exist_ok=True)

    for img_name in tqdm(img_names):
        src_img = img_dir / img_name
        if not src_img.exists():
            raise FileNotFoundError(f"Image {src_img} listed in {list_file} not found.")

        dst_img = root / "images" / split / img_name
        dst_img.parent.mkdir(parents=True, exist_ok=True)

        if not dst_img.exists():
            shutil.copy2(src_img, dst_img)

        ann_path = ann_dir / f"{Path(img_name).stem}.json"
        if not ann_path.exists():
            raise FileNotFoundError(f"Annotation {ann_path} missing for image {img_name}")

        data = json.load(open(ann_path))
        w, h = Image.open(src_img).size
        yolo_txt = convert_one(data, w, h)

        (labels_root / split / f"{Path(img_name).stem}.txt").write_text(yolo_txt)

print("\n✓ All splits converted.")

#data_seg.yaml
yaml_dict = {
    "path": str(root),
    "task": "segment",
    "train": "images/train",
    "val":   "images/val",
    "test":  "images/test",
    "nc": len(class_names),
    "names": class_names,
}
yaml_path = root / "data_seg.yaml"
yaml_path.write_text(yaml.dump(yaml_dict))
print(f"✓ data_seg.yaml saved to {yaml_path}")


Detected classes: ['Object']

▶ Preparing train split from train.json
train: 2320 images – first 5: ['9C2POCZ3HQ.jpg', 'KZ54GFWGXR.jpg', '3AOACRRDNX.jpg', 'TCMJNBOUCP.jpg', '8H8X5C0L47.jpg']


100%|█████████████████████████████████████| 2320/2320 [00:00<00:00, 3462.59it/s]



▶ Preparing val split from val.json
val: 496 images – first 5: ['XOY3JQ4X41.jpg', 'JN1OLSKJJ9.jpg', 'PTLSX8XT80.jpg', '3O5PQ3RLGE.jpg', 'MHKGO44MF7.jpg']


100%|███████████████████████████████████████| 496/496 [00:00<00:00, 3493.97it/s]



▶ Preparing test split from test.json
test: 507 images – first 5: ['4CROKS0MKE.jpg', '8M0V5N05F0.jpg', 'LEFTUCPEDV.jpg', 'J8JR3ZFQ5L.jpg', '4AYGRNDMQF.jpg']


100%|███████████████████████████████████████| 507/507 [00:00<00:00, 3210.42it/s]


✓ All splits converted.
✓ data_seg.yaml saved to /home/kudkud/Desktop/same-object-transfer-set/data_seg.yaml
